In [ ]:
# imports
import pickle
import os
import re 
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score 
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import sklearn.metrics as metrics
import scipy as sp
import scipy.stats as stats
import warnings
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from toeplitzlda.classification import ToeplitzLDA
import logging
import math

# utils functions
from utils.preprocessing import all_have_same_condition, list_iterations, load_complete_session, inspect_session, get_n_epochs, get_iterations, get_n_iterations, load_session_chached
from utils.feature_extraction import get_jumping_means, epoch_vectorizer_channelprime
from utils.offline_evaluation import compare_auc_single_trial_interval, compute_auc_with_cv
from utils.online_simulation import online_simulation

# Turn off warnings (that most likely occur from ToeplitzLDA)
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
mne.set_log_level('WARNING')



### Test session 3

In [ ]:
# Calibration/offline performance
# datapath = "data_p1/P1_S3/anonymized"
trials, _, _ = load_session_chached("data_p1/P1_S3/anonymized")
clf_ival_boundaries = np.array([0.1, 0.2, 0.3, 0.4, 0.5])

compare_auc_single_trial_interval(trials, 0, 12, test_size=0.2, ival_bounds=clf_ival_boundaries, plot_roc_curves=True) 
compute_auc_with_cv(trials,0,12,ival_bounds=clf_ival_boundaries,cv_folds=4,show_mean=True,show_folds=True)

# 08-05-2025 09:27 further optimizing, code cleanup
# timeit outcome
# 54.1 s ± 893 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
calibration_trials = trials[0:12]
online_trials = trials[12:22]
%time online_trial_targets = online_simulation(calibration_trials, online_trials, log_process="testtesttest.log")


### Test functions of LDA, sLDA, BTLDA class

In [ ]:
X_train = [[3,20],
     [2,7],
     [1,10],
     [6,24],
     [1,7],
     [8,19],
     [1,8],
     [5,21]]
X_train = np.array(X_train) 
y_train = np.array([1, 2, 2, 1, 2, 1, 2, 1])

print(np.cov(X_train.T))

lda = LDA()
ldaclf1 = lda.fit(X_train, y_train)

print(ldaclf1.classes_)
print(ldaclf1.coef_)

S_w = ((np.cov(X_train[y_train==1].T)) + np.cov(X_train[y_train==2].T))/2
w = np.dot (np.linalg.pinv(S_w) , (np.mean(X_train[y_train==2],axis=0))-np.mean(X_train[y_train==1],axis=0))

ldaclf = make_pipeline(LDA(),)
ldaclf.fit(X_train,y_train)
print("-----LDA-----")
print((ldaclf.get_params()))
print(ldaclf.get_params().keys())
print((ldaclf.get_params().get("steps")))
print((ldaclf.get_params().get("verbose")))
print((ldaclf.get_params().get("lineardiscriminantanalysis")).coef_) # obtain w

slda = make_pipeline(LDA(solver='lsqr', shrinkage='auto'),)
slda.fit(X_train,y_train)

print("-----SLDA------")
print(slda.get_params()) 
print(slda.get_params().get("lineardiscriminantanalysis").coef_) # obtain w
print(slda.get_params().get("lineardiscriminantanalysis").__dict__)

nch = 2
btlda = make_pipeline(ToeplitzLDA(n_channels=nch),)
btlda.fit(X_train,y_train)

print("-----BTLDA-----")
print((btlda.get_params().get("toeplitzlda")).coef_) # cov matrix
print(btlda.get_params().get("toeplitzlda").intercept_) # cov matrix
print(btlda.get_params().get("toeplitzlda").__dict__) # cov matrix




### Online adaptation sw function dump

In [ ]:
def online_adaptation_simulation_sw(raw_calibration_trials, online_trials, ival_bounds = np.array([0.1, 0.2, 0.3, 0.4, 0.5]), log_process=None):

    if log_process is not None:
        for handler in logging.root.handlers[:]:
            logging.root.removeHandler(handler)

        logging.basicConfig(
            filename=log_process,
            encoding="utf-8",
            filemode="w", # 'a' to not overwrite current log, 'w' to overwrite. This setting can be changed later
            level=logging.DEBUG)

        logging.info("New log file")


    clf_ival_boundaries = ival_bounds
    raw_calibration_trials = raw_calibration_trials

    ### Training the classifier ---------------------------------------------------------

    calibration_trials = [[get_jumping_means(iteration, clf_ival_boundaries) for iteration in trial] for trial in raw_calibration_trials]
    calibration_trials_reshaped = [[epochs.transpose(0, 2, 1) for epochs in trial] for trial in calibration_trials]
    calibration_stimuli = [epoch.reshape(-1) for trial in calibration_trials_reshaped for iteration in trial for epoch in iteration]
    calibration_stimuli = np.array(calibration_stimuli)
    calibration_labels = [(1 if event > 107 else 0) for trial in raw_calibration_trials for iteration in trial for event in iteration.events[:,2]] 
    calibration_labels = np.array(calibration_labels) 
    X_train = calibration_stimuli
    y_train = calibration_labels

    ### LDA
    ldaclf = make_pipeline(LDA(),)
    ldaclf.fit(X_train,y_train)

    ### Shrinkage LDA
    slda = make_pipeline(LDA(solver='lsqr', shrinkage='auto'),)
    slda.fit(X_train,y_train)

    ### BT-LDA
    nch = (raw_calibration_trials[0][0]).info["nchan"] # assuming nchan is constant in all trials
    btlda = make_pipeline(
        ToeplitzLDA(n_channels=nch),
    )
    btlda.fit(X_train,y_train)

    if log_process:
        logging.info("Trained all three classifiers on the calibration data.")
        logging.info("Online simulation starts")

    ### Online simulation ------------------------------------------------------------------

    # Extract relevant data, labels and the played words

    # Using list comprehension
    online_trial_targets = np.array([trial[0]["Target"].events[:,2][0] % 10 for trial in online_trials]) # The target word per trial
    online_labels = [(1 if event > 107 else 0) for trial in online_trials for iteration in trial for event in iteration.events[:,2]]            
    online_labels = np.array(online_labels) # conversion to np array is maybe not even needed
    online_words = [(iteration.events[:,2]%10) for trial in online_trials for iteration in trial]
    online_words = np.array(online_words) # conversion to np array is maybe not even needed

    # online_trial_targets: the target word per trial
    # print(len(online_trial_targets)) # e.g. 150
    # print(online_trial_targets[:10]) # e.g. [4 5 6 2 3 1 1 6 2 4]

    # online_labels: the location of the target word in the sequence of 6 stimuli per iteration. Note that the order of stimuli differs per iteration.
    # print(len(online_labels)) # e.g. 8244 = labels of all epochs (= n_trials * n_iterations per trial * n_epochs per iteration)
    # print(online_labels[:18]) # e.g. [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0]
   
    # #online_words: The word ID sequence that is presented per iteration. Note that the order of the presented words differs between iterations.
    # print(len(online_words)) # e.g. 1374
    #print(online_words[:6]) # e.g. [[1 6 2 3 5 4]
                             #       [6 3 1 2 4 5]
                             #       [6 3 5 2 1 4]
                             #       [3 1 2 6 4 5]
                             #       [1 6 5 2 4 3]
                             #       [5 4 1 6 2 3]]
    print(online_words.shape) # (n_trials * avg_n_iterations_p_trial, 6)

    if log_process:
        logging.info("Number of online trials: {}, which is {} runs".format(len(online_trials), len(online_trials)/6))


    # Predict target/non-target per stimulus.

    signed_distances_lda = np.zeros(len(online_labels))
    signed_distances_slda = np.zeros(len(online_labels))
    signed_distances_btlda = np.zeros(len(online_labels))

    count = 0 
    played_word_count = 0

    # word decision
    trial_predictions_lda = np.zeros(online_trial_targets.shape)
    trial_predictions_slda = np.zeros(online_trial_targets.shape)
    trial_predictions_btlda = np.zeros(online_trial_targets.shape)


    for t, trial in enumerate(online_trials):
        print("trial {}/{}".format(t, len(online_trials)))
        if log_process:
            logging.info("------------------ Run {} Trial {} ------------------".format(math.trunc(t/6)+1,t+1))
            logging.info("{epoch} \t| {word_id} \t| {LDA} \t| {SLDA} \t| {BTLDA} ")

        stim_distances_lda = np.zeros((len(trial),6))
        stim_distances_slda = np.zeros((len(trial),6))
        stim_distances_btlda = np.zeros((len(trial),6))

        for i, iteration in enumerate(trial):
            for s, stimulus in enumerate(iteration):
                s1 = (ldaclf.decision_function(get_jumping_means(iteration[s],clf_ival_boundaries).transpose(0,2,1).flatten().reshape(1,-1)))[0]
                signed_distances_lda[count] = s1 # Compute signed distance of stimulus to decision boundary

                s2 = (slda.decision_function(get_jumping_means(iteration[s],clf_ival_boundaries).transpose(0,2,1).flatten().reshape(1,-1)))[0]
                signed_distances_slda[count] = s2 # Compute signed distance of stimulus to decision boundary

                s3 = btlda.decision_function(get_jumping_means(iteration[s],clf_ival_boundaries).transpose(0,2,1).flatten().reshape(1,-1)).item()
                signed_distances_btlda[count] = s3 # Compute signed distance of stimulus to decision boundary
                
                if log_process:
                    logging.info("{} \t| {} \t| {} \t| {} \t| {}".format(count, iteration[s].events[:,2], s1, s2, s3))

                # for word decision
                played_word = online_words[played_word_count,s] - 1 # convert to index
                stim_distances_lda[i,played_word] = s1 # order computed distances according to word id
                stim_distances_slda[i,played_word] = s2
                stim_distances_btlda[i,played_word] = s3 

                ### adaptation (sliding window)
                
                new_single_epoch = get_jumping_means(iteration[s],clf_ival_boundaries).transpose(0,2,1).flatten().reshape(1,-1)
                new_single_label = online_labels[count]
                # update X_train and y_train data
                X_train = np.append(X_train,new_single_epoch, axis=0)
                X_train = X_train[1:]
                y_train = np.append(y_train,new_single_label)
                y_train = y_train[1:]
                # only apply this new training data to update our classifier after a single trial

                count+=1

                # Important note during debugging
                # btlda.decision_function returns an nd array of shape (). To access its value, you have to call .item() additionally, instead of taking the first element via [0] (as done for lda and slda)
            
            played_word_count += 1

        means_lda = np.mean(stim_distances_lda, axis=0) # get the mean distance for each word
        means_slda = np.mean(stim_distances_slda, axis=0) # get the mean distance for each word
        means_btlda = np.mean(stim_distances_btlda, axis=0) # get the mean distance for each word

        best_guess_lda = np.argmax(means_lda)
        best_guess_slda = np.argmax(means_slda)
        best_guess_btlda = np.argmax(means_btlda)

        # best_distances_lda = stim_distances_lda[:, best_guess_lda].flatten()
        # best_distances_slda = stim_distances_slda[:, best_guess_slda].flatten()
        # best_distances_btlda = stim_distances_btlda[:, best_guess_btlda].flatten()

        #not_best_distances = stim_distances[:,np.arange(stim_distances.shape[1])!=best_guess].flatten()
        #t_score, p = stats.ttest_ind(best_distances, not_best_distances, equal_var = False)

        trial_predictions_lda[t] = best_guess_lda + 1
        trial_predictions_slda[t] = best_guess_slda + 1
        trial_predictions_btlda[t] = best_guess_btlda + 1

        ### Adaptation: update our classifier after a trial has finished
        ## LDA
        ldaclf = make_pipeline(LDA(),)
        ldaclf.fit(X_train,y_train)
        ## Shrinkage LDA
        slda = make_pipeline(LDA(solver='lsqr', shrinkage='auto'),)
        slda.fit(X_train,y_train)
        ## BT-LDA
        nch = (raw_calibration_trials[0][0]).info["nchan"] # assuming nchan is constant in all trials
        btlda = make_pipeline(
            ToeplitzLDA(n_channels=nch),
        )
        btlda.fit(X_train,y_train)


        #print("Trial %d target prediction: word %d with p-value of %0.6f" % (t, best_guess+1, p)) 
        if log_process:
            logging.info("------------------ End of trial ------------------".format(math.trunc(t/6)+1,t+1))
            logging.info("{real_word} \t| {LDA_prediction} \t| {SLDA_prediction} \t| {BTLDA_prediction} ")
            logging.info("{} \t| {} \t| {} \t| {} ".format(online_trial_targets[t],best_guess_lda+1,best_guess_slda+1,best_guess_btlda+1))
            logging.info("Updated all three classifiers, this trial is now included in the training set")

    fpr, tpr, thresholds = metrics.roc_curve(online_labels,signed_distances_lda) 
    auc_fig = metrics.RocCurveDisplay(fpr=fpr, tpr = tpr)
    auc_fig.plot()
    plt.plot([0, 1],[0,1], '--')
    plt.legend(['ROC curve (area = %0.5f)' % metrics.auc(fpr, tpr), 'area = 0.5'], loc="lower right")
    plt.title("AUC-ROC Curve of the LDA classifier [online]")
    plt.show()


    fpr, tpr, thresholds = metrics.roc_curve(online_labels,signed_distances_slda) 
    auc_fig = metrics.RocCurveDisplay(fpr=fpr, tpr = tpr)
    auc_fig.plot()
    plt.plot([0, 1],[0,1], '--')
    plt.legend(['ROC curve (area = %0.5f)' % metrics.auc(fpr, tpr), 'area = 0.5'], loc="lower right")
    plt.title("AUC-ROC Curve of the sLDA classifier [online]")
    plt.show()


    fpr, tpr, thresholds = metrics.roc_curve(online_labels,signed_distances_btlda) 
    auc_fig = metrics.RocCurveDisplay(fpr=fpr, tpr = tpr)
    auc_fig.plot()
    plt.plot([0, 1],[0,1], '--')
    plt.legend(['ROC curve (area = %0.5f)' % metrics.auc(fpr, tpr), 'area = 0.5'], loc="lower right")
    plt.title("AUC-ROC Curve of the BT-LDA classifier [online]")
    plt.show()

    # word prediction
    
    # plot_distribution_comparison(not_best_distances, best_distances)
    print("Accuracy LDA: %0.5f" % np.mean(trial_predictions_lda == online_trial_targets))
    print("Accuracy SLDA: %0.5f" % np.mean(trial_predictions_slda == online_trial_targets))
    print("Accuracy BT-LDA: %0.5f" % np.mean(trial_predictions_btlda == online_trial_targets))

    if log_process:
        logging.info("Accuracy LDA: %0.5f" % np.mean(trial_predictions_lda == online_trial_targets))
        logging.info("Accuracy SLDA: %0.5f" % np.mean(trial_predictions_slda == online_trial_targets))
        logging.info("Accuracy BT-LDA: %0.5f" % np.mean(trial_predictions_btlda == online_trial_targets))

    return online_trial_targets


### LDA Class from scratch

In [ ]:
class MyLDA:
    "LDA with convex combination adaptation"

    def __init__(self):
        self.trained = False
        self.w = None
        self.b = None
        self.S1 = None # cov matrix target class
        self.S2 = None # cov matrix non target class
        self.S_w = None # within class cov matrix
        self.m1 = None
        self.m2 = None

    def manual_cov_matrix(self, X):
        """cov matrix from scratch
        
        inputs:
        X: data of dimensionality N x D
        
        Note that np.cov(X) accepts data of dimensionality D x N  !
        """

        m = np.mean(X,0)
        N = X.shape[0]
        X_mean_centered = X - m
        cov_matrix = 1/(N-1) * (np.dot(np.transpose(X_mean_centered),(X_mean_centered)))
        return cov_matrix   
         
    def fit(self, X, y):
        """LDA from scratch for binary classes
        
        inputs:
        - X: data of dimensionality N x D
        - y: labels of dimensionality N. 
            takes values {1,2}
        
        outputs: 
        - w: weight vector 
        - b: bias (scalar)
        """
        
        # Extract class data matrix 
        X_class_1 = X[y==1]
        X_class_2 = X[y==2]

        # Obtain the class means (take mean across samples N)
        m1 = np.mean(X_class_1,0) 
        m2 = np.mean(X_class_2,0)

        # Compute the cov matrix manually (see function manual_cov_matrix() above)
        S1 = self.manual_cov_matrix(X_class_1)
        S2 = self.manual_cov_matrix(X_class_2)

        # Compute the weight vector
        S_w = 0.5 * (S1 + S2)
        w = np.dot ( np.linalg.pinv(S_w) , m2 - m1)
        b = -0.5 * np.dot(w.T,(m1 + m2))

        self.m1 = m1
        self.m2 = m2
        self.S1 = S1
        self.S2 = S2
        self.S_w = S_w
        self.w = w
        self.b = b
        self.trained = True

    def decision_function(self, X):
        assert self.trained is True, "Classifier has not been trained yet"
        distance = np.dot(X,self.w.T) + self.b
        return distance
    
    
        
# Test whether my LDA works as sklearn's LDA
# Creating fake data X and y
X_train = [[3,20],
     [2,7],
     [1,10],
     [6,24],
     [1,7],
     [8,19],
     [1,8],
     [5,21]]
X_train = np.array(X_train) 

# X has dimensionality N x D
# print(X.shape) # (8,2)

# Corresponding labels (1: class 0, 2: class 1) 
# Note that the classes are of equal size
y_train = np.array([1, 2, 2, 1, 2, 1, 2, 1])

# Compute output of my LDA
myclf = MyLDA()
myclf.fit(X_train,y_train) 
print("Trained LDA from scratch")
print("Obtained w vector w = \t\t", myclf.w)
print("Obtained bias b = \t\t",myclf.b)   

# Testing decision_function()
x = [[2, 6],[1,7]]
y = [1,2]
print("MyLDA decision function: \t\t",myclf.decision_function(x))

# Compute output of sklearn's LDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA 
clf = LDA()
clf.fit(X_train, y_train)

print("\nTrained LDA from sklearn")
print("Obtained w vector w = \t\t",clf.coef_[0])
print("Obtained bias b = \t\t", clf.intercept_[0])
print("sklearn's LDA decision function: \t",clf.decision_function(x))